<a href="https://colab.research.google.com/github/MadisonJBollinger/lis5693/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sklearn

This is the package we are using for this lab.

In [ ]:
import requests
import io

url= "https://raw.githubusercontent.com/MadisonJBollinger/lis5693/refs/heads/main/lab-4/bbc-news-data.csv"
response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors
text = response.text

This is the dataset that was assigned to me and another. To fully read the dataset, I will now add Pandas so that it will read the items in the .csv.

In [ ]:
import pandas as pd

df = pd.read_csv(io.StringIO(text), sep='\t')
print(df.head())

I was receiving a ParserError, when I had Colab Gemini to explain the error it showed then added "sep='\t". When it did so, it fixed the issue

In [ ]:
df['category']

In [ ]:
df['title']

In [ ]:
df['content']

In [ ]:
features=df['content']
targets=df['category']

I did multiple dfs to view the given columns separately to gain a better understanding.
I chose to do the category column as the target label as that will provide a standard for what keywords best match from the text data to double-check my work.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

# split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(features, targets, test_size=0.3, random_state=42)

# Show the results of the split
print("Training set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))

# Remove or replace NaN values in X_train and X_test
X_train = X_train.fillna('')  # Replace NaN with empty string
X_test = X_test.fillna('')   # Replace NaN with empty string

preprocessing = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer())
])

print("Preprocessing training data...")
train_preprocessed = preprocessing.fit_transform(X_train)

print("Preprocessing test data...")
test_preprocessed = preprocessing.transform(X_test)

In [ ]:
y_test.unique()

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


nb_classifier = MultinomialNB()
svm_classifier = LinearSVC()
lr_classifier = LogisticRegression(multi_class="ovr")

print("Training Naive Bayes classifier...")
nb_classifier.fit(train_preprocessed, y_train)

print("Training SVM classifier...")
svm_classifier.fit(train_preprocessed, y_train)

print("Training Logistic Regression classifier...")
lr_classifier.fit(train_preprocessed, y_train)

In [ ]:
nb_predictions = nb_classifier.predict(test_preprocessed)
svm_predictions = svm_classifier.predict(test_preprocessed)
lr_predictions = lr_classifier.predict(test_preprocessed)

I chose the Naive Bayes, Logistic Regression, and Support Vector Machine as they would be best for such large data. I believe that Naive Bayes would be great for this, as it processes the large dataset.

In [ ]:

import numpy as np

print("NB Accuracy:", np.mean(nb_predictions == y_test))
print("SVM Accuracy:", np.mean(svm_predictions == y_test))
print("LR Accuracy:", np.mean(lr_predictions == y_test))

In [ ]:
target_names= ['business', 'sport', 'politics', 'entertainment', 'tech']

In [ ]:

from sklearn.metrics import classification_report  # Import classification_report

print(classification_report(y_test, nb_predictions, target_names=target_names))
print(classification_report(y_test, lr_predictions, target_names=target_names))
print(classification_report(y_test, svm_predictions, target_names=target_names))

Based on the classification from the three classifiers, SVM showed the best results, with a 0.96 to 0.99 f1-score and a 0.95 to 0.99 precision The other two classifiers provide mid to high 0.90s as well, but SVM is the highest for all the classifications.

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {'C': np.logspace(0, 3, 10)}
parameters = {'C': [0.1, 1, 10, 100, 1000]}

print("Grid search for SVM")
svm_best = GridSearchCV(svm_classifier, parameters, cv=10, verbose=1)
svm_best.fit(train_preprocessed, y_train)

In [ ]:
print("Best SVM Parameters")
print(svm_best.best_params_)

In [ ]:
best_svm_predictions = svm_best.predict(test_preprocessed)

print("Best SVM Accuracy:", np.mean(best_svm_predictions == y_test))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, svm_predictions, target_names=target_names))

In [ ]:
%matplotlib inline
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt

conf_matrix = confusion_matrix(y_test, svm_predictions)
conf_matrix_df = pd.DataFrame(conf_matrix, index=target_names, columns=target_names)

plt.figure(figsize=(15, 10))
x = sn.heatmap(conf_matrix_df, annot=True, vmin=0, vmax=conf_matrix.max(), fmt='d', cmap="YlGnBu")
plt.yticks(rotation=0)
plt.xticks(rotation=90)

The most incorrect labels were 5 politics when they were business. Other than that, 13 other mistakes were made, which is a great result.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/MadisonJBollinger/lis5693/main/lab-4/pseudo_dataset.csv"

pseudo_df = pd.read_csv(url)

pseudo_df.head()

I utilized ChatGPT to create a pseudo dataset as requested.

In [84]:
pseudo_preprocessed = preprocessing.transform(pseudo_df["text"].fillna(''))

In [85]:
predicted_labels = svm_classifier.predict(pseudo_preprocessed)

In [86]:
pseudo_preprocessed = preprocessing.transform(pseudo_df["text"].fillna(''))

In [89]:
pseudo_df["predicted_category"] = predicted_labels

In [90]:
print(pseudo_df.columns)
pseudo_df.head(10)

Index(['text', 'category', 'predicted_category'], dtype='object')


,text,category,predicted_category
0,The government announced a new policy aimed at...,NaN,business
1,A major technology firm unveiled its latest sm...,NaN,business
2,The national football team secured a dramatic ...,NaN,sport
3,Stock markets climbed after positive earnings ...,NaN,business
4,A popular actor received critical acclaim for ...,NaN,entertainment
5,Parliament debated a new education reform bill...,NaN,politics
6,A tennis champion advanced to the finals after...,NaN,sport
7,A startup company introduced a new artificial ...,NaN,business
8,Music fans celebrated the release of a long-aw...,NaN,entertainment
9,Oil prices rose following global supply concerns.,NaN,business


In [93]:
pseudo_df.to_csv("predicted.csv", index=False)

Checking the first fifteen predictions showed that 5 of the 15 were incorrectly labeled, though 2 of the 5 were understandable as they slipped me up before fulling realizing what it should be, line 1 and line 7. Other than that the rest looked great and showed good determination. I would most likely further keyword items to make it more accurate.

I believe that the given dataset was easy to use, as there is not many columns and it made decisions on testing easier. One of my dificulties was early in and had to use Google Gemini to catch my mistake and fix it to continue the process. Overall, the best part of it was working with the confusion matrix and seeing it in progress as it gave me a better understanding of what I see in the outputs than with previous work.